# Sensitivity studies

<div style="
    border: 1px solid #ddd;
    border-radius: 8px;
    padding: 12px 16px;
    max-width: 500px;
    background-color: #f9f9f9;
">
  <strong>✈️⚡ Electric Aircraft Design</strong><br>
  <br>
  <strong>Author:</strong> Florent LUTZ<br>
  <strong>Affiliation:</strong> ISAE-SUPAERO / DCAS <br>
  <strong>Course:</strong> Electric Aircraft Design<br>
</div>

For this study, the use of the Kodiak 100 hybridisation case presented in [3] is proposed. Besides the reasons discussed in the introductory Notebook, the Kodiak 100 was selected as a test case because of DAHER's recent involvement in the [TAGINE project](https://www.safran-group.com/pressroom/french-aerospace-consortium-launches-initiative-develop-hybrid-electric-propulsion-general-aviation-2025-06-18) which aims at integrating a hybrid powertrain on a General Aviation aircraft and their proposed use of the Kodiak 100 as a test platform. Additionally, the range of viability of electrified aircraft is predicted to be between 50 and 200 nm [1], which coincides very well with the actual use of the Kodiak 100 (see Figure 2 in Notebook [00_Introduction](00_Introduction.ipynb)).

Regarding the propulsive architecture, a parallel hybrid powertrain will be considered. The thermal branch will be identical to the original design while the electric branch will consist in a battery driven DC network. The architecture is presented in the next Figure.

<p align="center">
  <img src="./resources/hybrid_kodiak_100.png" width="900">
  <br>
  <em>Figure 4 : Propulsive architecture for the hybrid Kodiak 100</a></em>
</p>

To accelerate a potential EIS of the hybridised Kodiak and to reduce time to certification, a retrofit approach will be adopted. This means that no redesign of the airframe will be made, keeping the same as the original design. This will be modelled as a constant geometry and a constant airframe weight. Additionally, the payload will be adjusted in order to ensure that the MTOW of the hybridised Kodiak is equal to or below that of the reference aircraft. This means that if the weight of the powertrain gets high enough, the payload capabilities of the hybrid design will be reduced. For that, the following equation will be used:

$PL_{des} = min(PL_{max}, MTOW_{ref}-OWE-m_{fuel,des})$

Because of the technological maturity of today's electric components, range and performance reduction might be required for the feasible design of the hybrid aircraft. However, to set commercially realistic targets, we will pick a range following the results illustrated in Figure 2 in Notebook [00_Introduction](00_Introduction.ipynb). Looking at Figure 2, two thirds of the flights have range of 100 nm or less. Nonetheless, a range of 200 nm will be selected for the case of the hybridised Kodiak. While this is above the most frequently flown range, for a mission of 100 nm and considering the cruise we selected for the design mission of the original Kodiak 100, only half of the mission would be flown in cruise conditions according to the reference aircraft's Pilot Operating Handbook (POH). This range will serve as target for the hybridised version of the Kodiak 100. For comparison, we will evaluate the performances of the original Kodiak 100 on that same mission considering its maximum payload of 1140 kg. The aircraft will be assumed to be loaded with just enough fuel to perform the mission. This off-design evaluation will also be conducted using the FAST-OAD-CS23-HE framework which allows for on- and off-design performances evaluations. The same flight profile will be assumed, only the range and payload will be changed.

## Kodiak 100 off-design evaluation

The off-design evaluation of the Kodiak 100 will be done in the next cell. As this is an off-design evaluation the names of the quantity of interest will differ slightly from the variable names in [the introductory Notebook](00_Introduction.ipynb). Next cell will be prepared but you are expected to enter the data corresponding to the range and payload selected.

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
Because of the intricacies of the FAST-OAD-CS23-HE framework and although the propulsive architecture will be the same, the name of the powertrain description file will be slightly different. Its content remains identical though.
</div>

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The introductory Notebook needs to be run for this Notebook to work. Indeed results like the geometry, aerodynamics and some weights will be reused.
</div>

<div style="
  border-left: 4px solid #d32f2f;
  padding: 10px 12px;
  background-color: #fdecea;
  margin: 10px 0;
">
<strong>✏️ Action required</strong><br>
Modify the parameters left as ... in the <strong>next cell</strong> before executing it.
</div>

In [ ]:
%%capture

# Import useful packages
import pathlib
import shutil
import warnings
import fastoad.api as oad
from utils import rename_variables_for_payload_range

# Write the paths to the files of interest
DATA_FOLDER_PATH = "data"
RESULT_FOLDER_PATH = "results"

path_to_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "kodiak_100_configuration_off_design.yml"
path_to_design_output_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_final_output.xml"
path_to_off_design_input_file = pathlib.Path(DATA_FOLDER_PATH) / "kodiak_100_input_off_design.xml"
path_to_off_design_final_input_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_final_input_off_design.xml"

# Generate the final input file for the off-design computation, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    shutil.copy(path_to_design_output_file, path_to_off_design_input_file)
    rename_variables_for_payload_range(path_to_off_design_input_file)
    oad.generate_inputs(path_to_configuration_file, path_to_off_design_input_file, overwrite=True)

# This is where the value for the off-design mission should be inputted. The cell won't run until it is filled in.
off_design_mission_range = ...  # In nm
off_design_mission_payload = ... # In kg

off_design_mission_input_datafile = oad.DataFile(path_to_off_design_final_input_file)
off_design_mission_input_datafile["data:mission:operational:range"].value[0] = off_design_mission_range
off_design_mission_input_datafile["data:mission:operational:range"].units = "NM"
off_design_mission_input_datafile["data:mission:operational:payload:mass"].value[0] = off_design_mission_payload
off_design_mission_input_datafile["data:mission:operational:payload:mass"].units = "kg"
off_design_mission_input_datafile["data:mission:operational:payload:CG:x"].value[0] = 4.1
off_design_mission_input_datafile.save()

with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    kodiak_100_off_design_process = oad.evaluate_problem(path_to_configuration_file, overwrite=True)

Just like in the introductory Notebook, we can now look at the results for the mass and fuel consumption. FAST-OAD-CS23-HE also computes the emissions stemming from the combustion of the fuel for the mission and its production. In case the aircraft uses a battery, it also computes the emissions associated with the production of the electricity required to charge the battery. More specifically, it computes the equivalent warming potential of these species expressed as a function of the warming potential of CO2 (expressed in gCO2eq). Finally, to account for the aircraft's ability to transport passengers over a certain distance (severly hindered by the use of battery), it also computes this amount divided by each kg of payload and km travelled (in the realms of LCA this would be the functionnal unit). We will call this the emission factor. It's worth noting that this term can mean something different based on the context. These emission values will be used as metrics to compare the designs in future computations.

We can also use the tool presented in the previous Notebook to view the evolution of the aircraft's parameters during the flight. 

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
# Import useful packages
from tabulate import tabulate
from IPython.display import HTML, display
import openmdao.api as om
from fastga_he.gui.performances_viewer import PerformancesViewer

# Access the results with a datafile, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    kodiak_100_datafile = oad.DataFile(kodiak_100_off_design_process.output_file_path)

# Print the converted results with the actual value
cell_list = [
    ["Parameter", "Unit", "Computed value"], 
    ["Off-design mission TOW", "kg", om.convert_units(kodiak_100_datafile["data:mission:operational:TOW"].value[0], kodiak_100_datafile["data:mission:operational:TOW"].units, "kg")],
    ["Off-design mission fuel", "kg", om.convert_units(kodiak_100_datafile["data:mission:operational:fuel"].value[0], kodiak_100_datafile["data:mission:operational:fuel"].units, "kg")],
    ["Off-design mission emissions", "gCO2eq", om.convert_units(kodiak_100_datafile["data:environmental_impact:operational:emissions"].value[0], kodiak_100_datafile["data:environmental_impact:operational:emissions"].units, "g")],
    ["Off-design mission emission factor", "gCO2eq/kgPAX/km", kodiak_100_datafile["data:environmental_impact:operational:emission_factor"].value[0]],
]

display(HTML(tabulate(cell_list, tablefmt="html")))

# Define the path to the files containing mission data
path_to_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_propulsion_data_off_design.csv"
path_to_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "kodiak_100_mission_file_off_design.csv"

# Shows the evolution of the aircraft's parameter during the mission
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Look at the turboshaft power required throughout the mission and put a screenshot of it in your report. On that picture, showcase the maximum power and the cruise power. What can you say about the difference between the two ? <br>    
For future comparisons, write down the turboshaft's rated power.
</div>

## Hybrid Kodiak 100 (Hybridization scheme A)

The goal of this Notebook is to study the sensitivity of a hybrid electric design to a few select parameters. The first we will investigate is the hybridization scheme. In the first hybridization scheme we will consider, we will assume a constant hybridization ratio, meaning at each step of the flight X% of the power will come from the thermal branch and 1-X% from the electric branch. Let's start by investigating the influence of the hybridization ratio on the aircraft's metrics.

<div style="
  border-left: 4px solid #d32f2f;
  padding: 10px 12px;
  background-color: #fdecea;
  margin: 10px 0;
">
<strong>✏️ Action required</strong><br>
Modify the parameters left as ... in the <strong>next cell</strong> before executing it. Try different values of the hybridization ratio between 75% and 95% and mark down the results. They will be used in a future cell for post processing.
</div>

In [ ]:
path_to_hybrid_A_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "hybrid_kodiak_100_scheme_A_configuration.yml"
path_to_hybrid_A_input_file = pathlib.Path(DATA_FOLDER_PATH) / "hybrid_kodiak_100_scheme_A_input.xml"
path_to_hybrid_A_final_input_file = pathlib.Path(RESULT_FOLDER_PATH) / "hybrid_kodiak_100_scheme_A_final_input.xml"

# Generate the final input file for the hybrid design computation, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    oad.generate_inputs(path_to_hybrid_A_configuration_file, path_to_hybrid_A_input_file, overwrite=True)
    
# This is where the value for the hybridization ratio should be inputted. The cell won't run until it is filled in.
hybridization_ratio = ...  # In percent, i.e. 80.0, 90.0, ...

hybrid_A_input_datafile = oad.DataFile(path_to_hybrid_A_final_input_file)
hybrid_A_input_datafile["data:propulsion:he_power_train:planetary_gear:planetary_gear_1:power_split"].value[0] = hybridization_ratio
hybrid_A_input_datafile["data:propulsion:he_power_train:planetary_gear:planetary_gear_1:power_split"].units = "percent"
hybrid_A_input_datafile.save()

# Now we run the process
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    hybrid_kodiak_100_scheme_A_process = oad.evaluate_problem(path_to_hybrid_A_configuration_file, overwrite=True)

# Access the results with a datafile, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    hybrid_A_output_datafile = oad.DataFile(hybrid_kodiak_100_scheme_A_process.output_file_path)
    
# Look at the results
cell_list = [
    ["Parameter", "Unit", "Computed value"], 
    ["MTOW", "kg", om.convert_units(hybrid_A_output_datafile["data:weight:aircraft:MTOW"].value[0], hybrid_A_output_datafile["data:weight:aircraft:MTOW"].units, "kg")],
    ["OWE", hybrid_A_output_datafile["data:weight:aircraft:OWE"].units, hybrid_A_output_datafile["data:weight:aircraft:OWE"].value[0]],
    ["Payload", hybrid_A_output_datafile["data:weight:aircraft:payload"].units, hybrid_A_output_datafile["data:weight:aircraft:payload"].value[0]],
    ["Propulsive system weight", hybrid_A_output_datafile["data:propulsion:he_power_train:mass"].units, hybrid_A_output_datafile["data:propulsion:he_power_train:mass"].value[0]],
    ["Battery weight", hybrid_A_output_datafile["data:propulsion:he_power_train:battery_pack:battery_pack_1:mass"].units, hybrid_A_output_datafile["data:propulsion:he_power_train:battery_pack:battery_pack_1:mass"].value[0]],
    ["Turboshaft maximum power", hybrid_A_output_datafile["data:propulsion:he_power_train:turboshaft:turboshaft_1:power_rating"].units, hybrid_A_output_datafile["data:propulsion:he_power_train:turboshaft:turboshaft_1:power_rating"].value[0]],
    ["Electric motor maximum power", hybrid_A_output_datafile["data:propulsion:he_power_train:PMSM:motor_1:shaft_power_rating"].units, hybrid_A_output_datafile["data:propulsion:he_power_train:PMSM:motor_1:shaft_power_rating"].value[0]],
    ["Emissions from fuel", hybrid_A_output_datafile["data:environmental_impact:sizing:fuel_emissions"].units, hybrid_A_output_datafile["data:environmental_impact:sizing:fuel_emissions"].value[0]],
    ["Emissions from electricity", hybrid_A_output_datafile["data:environmental_impact:sizing:energy_emissions"].units, hybrid_A_output_datafile["data:environmental_impact:sizing:energy_emissions"].value[0]],
    ["Emissions factor", hybrid_A_output_datafile["data:environmental_impact:sizing:emission_factor"].units, hybrid_A_output_datafile["data:environmental_impact:sizing:emission_factor"].value[0]],
]

print("\n")

display(HTML(tabulate(cell_list, tablefmt="html")))

print("\n")

# Define the path to the files containing mission data
path_to_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "hybrid_kodiak_100_scheme_A_propulsion_data.csv"
path_to_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "hybrid_kodiak_100_scheme_A_mission_file.csv"

# Shows the evolution of the aircraft's parameter during the mission
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

While only a few outputs of the sizing process are highlighted in the previous cell, you can explicitly access all of them using the next cell. Be careful however: the next cell will only print out the results of the latest execution of the previous cell. If you want to change the hybridization ratio, re-run the previous cell to update the results in the next cell.

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
oad.variable_viewer(hybrid_kodiak_100_scheme_A_process.output_file_path)

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Using the graph of the propeller shaft power required, illustrate the hybridization scheme by showing the contribution of the thermal branch and the electric branch.<br>
Now, take a look at the active constraint for the battery pack. More specifically look the variables named <em>data:propulsion:he_power_train:battery_pack:battery_pack_1:c_rate_multiplier</em> and <em>data:propulsion:he_power_train:battery_pack:battery_pack_1:capacity_multiplier</em> and see which one is at 1. This is the active constraint.
</div>

Use the cell below to illustrate the variation of the parameters you've noted down in your calculation as a function of the hybridization ratio.

<div style="
  border-left: 4px solid #d32f2f;
  padding: 10px 12px;
  background-color: #fdecea;
  margin: 10px 0;
">
<strong>✏️ Action required</strong><br>
Modify the parameters left as ... in the <strong>next cell</strong> before executing it. You can also customize the figure for a clearer display.
</div>

In [ ]:
# Import useful packages
import plotly.graph_objects as go

x_data = [..., ..., ...] # [..., ..., ...]
y_data = [..., ..., ...] # [..., ..., ...]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=x_data, 
        y=y_data
    )
)
fig.update_layout(
    height=600,
    width=900,
)
fig.show()

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Plot and comment about the evolution of: the total emissions, the emissions from fuel, the emissions from electricity, the battery mass, the payload mass and the emission factor. Conclude about this hybridization scheme.<br>
For future comparison, write down the variation of the turboshaft's and electric motor's rated power.
</div>

It is important to recall that, for this hybridization scenario, a retrofit approach was used. Therefore the MTOW is limited and cannot diverge. An interesting exercise could be to approach this hybridization scheme with a design from scratch approach. The MTOW would be free to vary and could thus potentially rapidly increase.

## Hybrid Kodiak 100 (Hybridization scheme B)

The second hybridization scheme we will study is a configuration which aims to nullify the emissions at low altitude, particularly at the beginning of climb and at the end of descent. The reason for this interest is detailed in [2]:
> An important aspect to be analysed is the emission of the take-off and landing (LTO) phase, which includes the taxiing, take-off and climb phase up to an altitude of 915 m. This aspect is of particular interest for the improvement of the so-called local air quality, i.e. the air quality in the surrounding areas of the airport

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
The original electric motor model is calibrated with data from the <a href="https://emrax.com/e-motors/">EMRAX family</a> and works well for rated power below approximately 150 kW. Because of the hybridization scheme we will consider here, however, the expected shaft power is well above that limit. Therefore, we will use a slightly different model based on a different motor architecture. This architecture, proposed by the HASTECS project, is presented <a href="https://fast-oad-cs23-he.readthedocs.io/en/latest/documentation/models/propulsion/loads/smpmsm/index.html">here</a>. This motor allows for much higher rated power but functions only at high rpm (>10000 rpm). Therefore, we will also add a gearbox to increase the rotation speed selected for the shaft of the propeller (around 2000 rpm) to a suitable speed for the electric motor.
</div>

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
path_to_hybrid_B_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "hybrid_kodiak_100_scheme_B_configuration.yml"
path_to_hybrid_B_input_file = pathlib.Path(DATA_FOLDER_PATH) / "hybrid_kodiak_100_scheme_B_input.xml"

# Generate the final input file for the hybrid design computation, ignoring all useless warnings
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    oad.generate_inputs(path_to_hybrid_B_configuration_file, path_to_hybrid_B_input_file, overwrite=True)

# Now we run the process
with warnings.catch_warnings():
    warnings.simplefilter(action="ignore")
    hybrid_kodiak_100_scheme_B_process = oad.evaluate_problem(path_to_hybrid_B_configuration_file, overwrite=True)

Use the cell below to look at the evolution of the power in the electric branch and in the thermal branch. 

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Using the graph of the propeller shaft power required, illustrate the hybridization scheme by showing the contribution of the thermal branch and the electric branch.<br>
</div>

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
path_to_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "hybrid_kodiak_100_scheme_B_propulsion_data.csv"
path_to_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "hybrid_kodiak_100_scheme_B_mission_file.csv"

# Shows the evolution of the aircraft's parameter during the mission
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
You can see a discontinuity in the propeller shaft power profile. This is due to the fact that, when the turboshaft is active, the release of the hot gas as a result of the combustion produces some thrust. Therefore, the thrust the propeller is required to produce is reduced. A clear discontinuity can thus be seen when the turboshaft is "turned on".
</div>

The network viewer can also be used as a post-processing tool and provides an alternate way of inspecting the state of the powertrain at each step of the flight. To do so, however, requires the setting up of a local server to provide some animation. Confirm, by looking at the animation, that, at some point of the climb (or descent) only the electric branch is active. At an ulterior point, confirm that only the thermal branch is active. Note that the thermal branch will always have some residual power because it is runnning at idle even when not providing propulsive power. 

The launching of the animation can cause some interference with the Kernel this Notebook is using to run. As such we will need to set it up using an alternative script. Create a new launcher by going into "File" > "New Launcher" (or press Ctrl + Shift + L). Then, create a Python Console and run the following line.
> run animation_hybrid_kodiak_100_scheme_B.py

As usual, you can use FAST-OAD post-processing functions to inspect the values of the converged aircraft. This is done in the next cell.

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
oad.variable_viewer(hybrid_kodiak_100_scheme_B_process.output_file_path)

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
For future comparisons, write down the turboshaft's and electric motor's rated power.<br>
Comment about the value of the power density of the electric motor for hybridization scheme A and B. Conclude about the interest of this new motor architecture.<br>
Write down the value of the allowable payload for that configuration, conclude about the viability of this hybridization scheme <em> as is</em>.<br>
Identify the biggest contributor to the mass of the powertrain. For that component, identify the most constraining sizing criteria in that simulation. What can you propose to solve that issue ?
</div>

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell is hidden by default because it provides the answer to the previous question. You are encouraged to answer the question before looking at the next cell. The cell can be shown by clicking the small "..." icon on the left.
</div>

As you can see in the previous case, the battery is sized according to the power it needs to provide (actually it is sized based on the C-rate but this is equivalent). This is due to the hybridization scheme requiring high power demand on a relatively short period of time, thus requiring a small amount of energy overall. The cell which is used by default is a Samsung INR18650-35E which has a relatively high energy density and small power density as opposed to other cell in the Samsung INR18650 family. The characteristics of a few of the cells in that family are shown in Figure 5 below.

<p align="center">
  <img src="./resources/samsung_inr18650_cell_family.svg" width="1200">
  <br>
  <em>Figure 5 : Pseudo Ragone plot of the battery cells in the Samsung INR18650 family</em>
</p>

To investigate the influence of the choice of battery cell on the final design, the same hybridization scheme will be investigated with a different cell. For this next process, the Samsung INR18650-25R will be used.

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
Running the next cell will overwrite the results of the previous cell. You can always get the results of the previous cell again by re-running it. You can also rename the output file to avoid it being over-written. <br>
Changing the reference cell in the FAST-OAD-CS23-HE framework requires the process to be run in a slightly different manner. As such, the next cell will be different from what you have seen so far. However, it is, in essence, the exact same procedure.
</div>

<div style="
  border-left: 4px solid #f9a825;
  padding: 10px 12px;
  background-color: #fff8e1;
  margin: 10px 0;
">
The next cell should not be changed.
</div>

In [ ]:
with warnings.catch_warnings():
    hybrid_kodiak_100_scheme_B_process_configurator = oad.FASTOADProblemConfigurator(path_to_hybrid_B_configuration_file)
    hybrid_kodiak_100_scheme_B_process = hybrid_kodiak_100_scheme_B_process_configurator.get_problem()
    hybrid_kodiak_100_scheme_B_process.write_needed_inputs(path_to_hybrid_B_input_file)
    hybrid_kodiak_100_scheme_B_process.read_inputs()

    # Change battery pack characteristics so that they match those of a high power,
    # lower capacity cell like the Samsung INR18650-25R. Assumes same polarization curve
    hybrid_kodiak_100_scheme_B_process.model_options["*"] = {
        "cell_capacity_ref": 2.5,
        "cell_weight_ref": 45.0e-3,
        "reference_curve_current": [500, 5000, 10000, 15000, 20000],
        "reference_curve_relative_capacity": [1.0, 0.97, 1.0, 0.97, 0.95],
    }
    hybrid_kodiak_100_scheme_B_process.setup()
    hybrid_kodiak_100_scheme_B_process.set_val(
        "data:propulsion:he_power_train:battery_pack:battery_pack_1:cell:c_rate_caliber",
        val=8.0,
        units="h**-1",
    )
    hybrid_kodiak_100_scheme_B_process.run_model()
    hybrid_kodiak_100_scheme_B_process.write_outputs()

oad.variable_viewer(hybrid_kodiak_100_scheme_B_process.output_file_path)

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Write down the value of the allowable payload for that configuration and that cell.<br>
Write down the value of the emissions factor for that configuration. Conclude about the interest of that hybridization scheme, while keeping in mind that it allows no emissions at low altitude. <br>
Comment the evolution of the battery mass and the sizing constraints. Suggest an avenue of future improvement for that scheme.
</div>

## Hybrid Kodiak 100 (Hybridization scheme C)

The third and last hybridization scheme we will study is the configuration proposed in  [3]. This configuration proposes to reduce the rated power of the engine and complement it with power from an electric branch, often called a battery-boosted turboprop. The reason behind this choice is that it would allow the turboshaft to be sized for cruise power and thus operate closer to its optimal point, consequently reducing emissions with only mild hybridisation levels. This stems from the observation on the reference case that the rated power of the engine is significantly different than the cruise power, thus creating inefficiencies in the design. In practice, this means that instead of handling the hybridization ratio as a percentage, it will be handled as a threshold value. All power below that threshold value (corresponding to the turboshaft rated power) will be directed to the thermal branch and the rest will be directed to the electric branch. In FAST-OAD-CS23-HE this is called the power share mode. The difference between the two scheme is given in Figure 6. Conceptually, the thermal branch will handle the "continuous" part of the power demand and the electric branch the "transient part" or "high power parts".

<p align="center">
  <img src="./resources/splitter.png" width="900">
  <br>
  <em>Figure 6 : Operating principle of the different hybridization modes in FAST-OAD-CS23-HE</a></em>
</p>

As for the previous case, we will propose investigating the sensitivity of the design with respect to a few different battery cells.

<div style="
  border-left: 4px solid #d32f2f;
  padding: 10px 12px;
  background-color: #fdecea;
  margin: 10px 0;
">
<strong>✏️ Action required</strong><br>
In the <strong>next cell</strong>, the snippets of code required to test each cells are marked with a lign of "==========" symbol above and below them. To run the sizing process with one specific cell, uncomment the corresponding snippet of code. Make sure at only one block is uncommented for all execution of the cell.
</div>

In [ ]:
# Write the paths to the files of interest
path_to_hybrid_C_configuration_file = pathlib.Path(DATA_FOLDER_PATH) / "hybrid_kodiak_100_scheme_C_configuration.yml"
path_to_hybrid_C_input_file = pathlib.Path(DATA_FOLDER_PATH) / "hybrid_kodiak_100_scheme_C_input.xml"
path_to_powertrain_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "hybrid_kodiak_100_scheme_C_propulsion_data.csv"
path_to_mission_data_file = pathlib.Path(RESULT_FOLDER_PATH) / "hybrid_kodiak_100_scheme_C_mission_file.csv"

with warnings.catch_warnings():
    hybrid_kodiak_100_scheme_C_process_configurator = oad.FASTOADProblemConfigurator(path_to_hybrid_C_configuration_file)
    hybrid_kodiak_100_scheme_C_process = hybrid_kodiak_100_scheme_C_process_configurator.get_problem()
    hybrid_kodiak_100_scheme_C_process.write_needed_inputs(path_to_hybrid_C_input_file)
    hybrid_kodiak_100_scheme_C_process.read_inputs()
    
    # ==================== Uncomment the snippet below if you want to use the Samsung INR18650-35E ==================== #
    hybrid_kodiak_100_scheme_C_process.setup()
    # ==================== Uncomment the snippet above if you want to use the Samsung INR18650-35E ==================== #
    
    # ==================== Uncomment the snippet below if you want to use the Samsung INR18650-25R ==================== #
    # hybrid_kodiak_100_scheme_C_process.model_options["*"] = {
    #     "cell_capacity_ref": 2.5,
    #     "cell_weight_ref": 45.0e-3,
    #     "reference_curve_current": [500, 5000, 10000, 15000, 20000],
    #     "reference_curve_relative_capacity": [1.0, 0.97, 1.0, 0.97, 0.95],
    # }
    # hybrid_kodiak_100_scheme_C_process.setup()
    # hybrid_kodiak_100_scheme_C_process.set_val(
    #     "data:propulsion:he_power_train:battery_pack:battery_pack_1:cell:c_rate_caliber",
    #     val=8.0,
    #     units="h**-1",
    # )
    # ==================== Uncomment the snippet above if you want to use the Samsung INR18650-25R ==================== #
    
    # ==================== Uncomment the snippet below if you want to use the Samsung INR18650-30Q ==================== #
    # hybrid_kodiak_100_scheme_C_process.model_options["*"] = {
    #     "cell_capacity_ref": 2.950,
    #     "cell_weight_ref": 48.0e-3,
    #     "reference_curve_current": [600, 5000, 10000, 15000, 20000],
    #     "reference_curve_relative_capacity": [1.0, 0.97, 1.0, 0.97, 0.95],
    # }
    # hybrid_kodiak_100_scheme_C_process.setup()
    # hybrid_kodiak_100_scheme_C_process.set_val(
    #     "data:propulsion:he_power_train:battery_pack:battery_pack_1:cell:c_rate_caliber",
    #     val=5.0,
    #     units="h**-1",
    # )
    # ==================== Uncomment the snippet above if you want to use the Samsung INR18650-30Q ==================== #
    
    hybrid_kodiak_100_scheme_C_process.run_model()
    hybrid_kodiak_100_scheme_C_process.write_outputs()

oad.variable_viewer(hybrid_kodiak_100_scheme_C_process.output_file_path)

# Shows the evolution of the aircraft's parameter during the mission
perfo_viewer = PerformancesViewer(
    power_train_data_file_path=path_to_powertrain_data_file.as_posix(),
    mission_data_file_path=path_to_mission_data_file.as_posix(),
    plot_height=800,
    plot_width=1200,
)

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Evaluate the aircraft with this hybridization scheme using the 3 cells. In particular, write down and comment on: the allowable payload, the battery mass, the battery constraints and the emission factor. Select the best cell among those available. <br>
Given the results to the previous questions, use the Ragone plot of the Samsung INR18650 family and locate where would the "ideal" cell for that hybridzation scheme be located.<br>
Using the graph of the propeller shaft power required, illustrate the hybridization scheme by showing the contribution of the thermal branch and the electric branch. Additionally, show where the value of the turboshaft rated power is located. Comment on whether the original idea for that hybridization scheme is respected. <br>
Compare the value of the turboshaft rated power and electric motor power with all the other hybridization scheme and the reference case. 
</div>

Just like for the hybridization scheme B, you can use the network viewer as a post processing tool as an additional way to ensure that the hybridization scheme we selected is indeed being used. In a Python console run the command:
> run animation_hybrid_kodiak_100_scheme_C.py

Check that during the climb, both branches are active and that the electric branch turns off during cruise, descent and reserve.

## Conclusion

<div style="
  border-left: 4px solid #6a1b9a;
  padding: 10px 12px;
  background-color: #f3e5f5;
  margin: 10px 0;
">
<strong>📝 Report question</strong><br>
Based on the results you have obtained (mainly the values of emission factor), select the best configuration for the mission: reference aircraft, hybrid scheme A, hybrid scheme B or hybrid scheme C.
</div>

<div style="
  border-left: 4px solid #2b7de9;
  padding: 10px 12px;
  background-color: #f2f6fc;
  margin: 10px 0;
">
<strong>ℹ️ Note</strong><br>
While this case study is very telling, it works very well only because there are a lot of "inefficiencies" in the original design due to the low cruise speed selected and, in consequence, the low power required in cruise. The Kodiak 100 is actually typically used with higher speed when it does commuter missions. This also begs the question of a critical thinking of the aircraft's uses and when they are best suited for hybridization. For instance here, if we want to keep the low speed, the resulting aircraft could be useful for mission with high endurance like surveillance missions.
</div>

## Bibliography

[1] Langford, John S., and David K. Hall. "Electrified aircraft propulsion." The Bridge 50.2 (2020): 21-27.

[2] Abu Salem, K., and G. Palaia. "Environmental implications of hybrid-electric regional aircraft: emissions and climate change." 34th Congress of the International Council of the Aeronautical Sciences, ICAS 2024. International Council of the Aeronautical Sciences, 2024.

[3] Lutz, Florent, et al. "Study of hybridisation scenarios for turboprop aircraft in the general aviation segment." 34th Congress of the International Council of the Aeronautical Sciences, Florence, Italy. 2024.